# Segundo Projeto: Analisador Sintático

O segundo projeto requer que você implemente um analisador sintático para a linguagem uZig e gere uma árvore de sintaxe como saída (observe que a árvore de sintaxe abstrata será construída apenas no terceiro projeto). Para concluir este segundo projeto, você também pode usar o [SLY](https://sly.readthedocs.io/), uma versão Python do conjunto de ferramentas [lex/yacc](http://dinosaur.compilertools.net/) com a mesma funcionalidade mas com uma interface mais amigável. Por favor, leia o conteúdo completo desta seção e complete cuidadosamente as etapas indicadas.


## Visão Geral do Analisador Sintático: Construíndo uma Árvore de Sintaxe

Considere a seguinte gramática:

```
<program> ::= <statements> EOF

<statements> ::= { <statement> }+

<statement> ::= <print_statement>
              | <assign_statement>
              | <if_statement>

<print_statement> ::= PRINT <expression> SEMI

<assign_statement> ::= <identifier> EQUALS <expression> SEMI

<if_statement> ::= IF <expression> LBRACE <statements> RBRACE { ELSE LBRACE <statements> RBRACE }?

<expression> ::= <primary_expression> { (PLUS|TIMES|EQ|NE) <primary_expression> }*

<primary_expression> ::= <constant>
                       | <identifier>
                       | LPAREN <expression> RPAREN

<identifier> ::= ID

<constant> ::= NUM
```

Nessa gramática, a seguinte sintaxe é usada:

```
 {símbolos}*  ==> Zero ou mais repetições de símbolos
 {símbolos}+  ==> Um ou mais repetições de símbolos
 {símbolos}?  ==> Zero ou uma ocorrência de símbolos (opcional)
 sim1 | sim2  ==> Ou sim1 ou sim2 (uma escolha)
```

Alguns exemplos de sentenças válidas para essa gramática são:

```cpp
a = 3 + 4 * 5 ;
```

```cpp
print a ;
```

```cpp
a = 3;
b = 4 * a;
print (a+b);
```

```cpp
a = 7;
b = 10;
c = 5;

if a*a == b*b + c*c {
   print 1;
} else {
   print 0;
}
```

### Analisador Léxico

O primeiro passo é construir um analisador léxico para os terminais dessa gramática:

In [ ]:
# Class that represents a token
class Token:
    def __init__(self, type, value, lineno, index, end):
        self.type = type
        self.value = value
        self.lineno = lineno
        self.index = index
        self.end = end

    def __repr__(self):
        return f'Token(type={self.type!r}, value={self.value!r}, lineno={self.lineno}, index={self.index}, end={self.end})'

_literals = {
    # Operators
    '+': "PLUS",
    '*': "TIMES",
    '==': "EQ",
    '!=': "NE",
    # Assignment
    '=': "EQUALS",
    # Delimeters
    '(': "LPAREN",
    ')': "RPAREN",  # ( )
    '{': "LBRACE",
    '}': "RBRACE",  # { }
    ';': "SEMI",    # ;
    }

_keywords = {
    # Reserved keywords
    'print': "PRINT",
    'if': "IF",
    'else': "ELSE"
    }

# High level function that takes input source text and turns it into tokens.
# This is a natural place to use some kind of generator function.

def tokenize(text):
    index = 0
    lineno = 1
    size = len(text)
    while index < size:
        if text[index] in ' \t':
            index += 1
            continue

        elif text[index] == '\n':
            index += 1
            lineno += 1
            continue

        elif text[index].isdigit():
            start = index
            while index < size and text[index].isdigit():
                index += 1
            yield Token('NUM', text[start:index], lineno, start, index)
            continue

        elif text[index].isalpha():
            start = index
            while index < size and text[index].isalnum():
                index += 1
            tokvalue = text[start:index]
            if tokvalue in _keywords:
                yield Token(_keywords[tokvalue], tokvalue, lineno, start, index)
            else:
                yield Token('ID', tokvalue, lineno, start, index)
            continue

        elif text[index:index+2] in _literals:
            yield Token(_literals[text[index:index+2]], text[index:index+2], lineno, index, index+2)
            index += 2
            continue

        elif text[index] in _literals:
            yield Token(_literals[text[index]], text[index], lineno, index, index+1)
            index += 1
            continue

        else:
            print(f'{lineno}: Illegal character {text[index]!r}')
            index += 1

In [ ]:
def main(args):
    if len(args) > 0:
        for tok in tokenize(args[0]):
            print(tok)

In [ ]:
main(["a = 3 + 4 * 5 ;"])

### Reconhecendo as sentenças

Agora, vamos desenvolver um analisador sintático para reconhecer sentenças dessa gramática e construir uma árvore de sintaxe para elas.

In [ ]:
# Fake token to signal "end of file"
class EOF:
    type = 'EOF'
    value = 'EOF'
    lineno = 'EOF'
    index = -1
    end = -1

class Tokens:

    def __init__(self, tokens):
        self.tokens = tokens
        self._lookahead = None

    @property
    def lookahead(self):
        if self._lookahead is None:
            self._lookahead = next(self.tokens, EOF)    # <<< Fail at the "end of file"
        return self._lookahead

    def peek(self, *types):
        # Returns the current token, but without consuming it.
        if self.lookahead.type in types:
            return self.lookahead
        else:
            return None

    def accept(self, *types):
        # Accept the next token if it has a given type, otherwise return None (not an error)
        tok = self.peek(*types)
        if tok:
            self._lookahead = None
        return tok

    def expect(self, *types):
        # Requires the next token to have toktype or SyntaxError
        tok = self.accept(*types)
        if not tok:
            self.error()
        return tok

    def error(self):
        # Default error handling function.
        token = self.lookahead
        if token:
            lineno = getattr(token, 'lineno', 0)
            if lineno:
                raise SyntaxError(f'Syntax error at line {lineno}, token={token.type}\n')
            else:
                raise SyntaxError(f'Syntax error, token={token.type}')
        else:
            raise SyntaxError('Parse error in input. EOF\n')

# <program> ::= <statements> EOF
def parse_program(tokens):
    parse_statements(tokens)

# <statements> ::= { <statement> }+
def parse_statements(tokens):
    parse_statement(tokens)
    while tokens.peek('PRINT', 'ID', 'IF'):
        parse_statement(tokens)

# <statement> ::= <print_statement>
#               | <assign_statement>
#               | <if_statement>
def parse_statement(tokens):
    if tokens.peek('PRINT'):
        parse_print_statement(tokens)
    elif tokens.peek('ID'):
        parse_assignment_statement(tokens)
    elif tokens.peek('IF'):
        parse_if_statement(tokens)
    else:
        tokens.error()

# <print_statement> ::= PRINT <expression> SEMI
def parse_print_statement(tokens):
    tokens.expect('PRINT')
    parse_expression(tokens)
    tokens.expect('SEMI')

# <assign_statement> ::= <identifier> EQUALS <expression> SEMI
def parse_assignment_statement(tokens):
    parse_identifier(tokens)
    tokens.expect('EQUALS')
    parse_expression(tokens)
    tokens.expect('SEMI')

# <if_statement> ::= IF <expression> LBRACE <statements> RBRACE { ELSE LBRACE <statements> RBRACE }?
def parse_if_statement(tokens):
    tokens.expect('IF')
    parse_expression(tokens)
    tokens.expect('LBRACE')
    parse_statements(tokens)
    tokens.expect('RBRACE')

    if tokens.accept('ELSE'):
        tokens.expect('LBRACE')
        parse_statements(tokens)
        tokens.expect('RBRACE')

# <expression> ::= <primary_expression> { (PLUS|TIMES|EQ|NE) <primary_expression> }*
def parse_expression(tokens):
    parse_primary_expression(tokens)
    while tokens.accept('PLUS', 'TIMES', 'EQ', 'NE'):
        parse_primary_expression(tokens)

# <primary_expression> ::= <constant>
#                        | <identifier>
#                        | LPAREN <expression> RPAREN
def parse_primary_expression(tokens):
    if tokens.peek('NUM'):
        parse_constant(tokens)
    elif tokens.peek('ID'):
        parse_identifier(tokens)
    elif tokens.accept('LPAREN'):
        parse_expression(tokens)
        tokens.expect('RPAREN')
    else:
        tokens.error()

# <constant> ::= NUM
def parse_constant(tokens):
    tokens.expect('NUM')

# <identifier> ::= ID
def parse_identifier(tokens):
    tokens.expect('ID')

# High level function that takes input tokens and turns it into a syntax tree.
# This is a natural place to use some kind of generator function.

def parse_tokens(raw_tokens):
    try:
        tokens = Tokens(raw_tokens)
        return parse_program(tokens)
    except SyntaxError as e:
        print(e)

In [ ]:
# Top-level function that runs everything
def parse_source(text):
    tokens = tokenize(text)
    model = parse_tokens(tokens)     # You need to implement this part
    return model

In [ ]:
def build_tree(root):
    return '\n'.join(_build_tree(root))

def _build_tree(node):
    if isinstance(node, list):
        if not node: return
        node = tuple(node)

    if not isinstance(node, tuple):
        yield " "+str(node)
        return

    values = [_build_tree(n) for n in node]
    if len(values) == 1:
        yield from build_lines('──', '  ', values[0])
        return

    start, *mid, end = values
    yield from build_lines('┬─', '│ ', start)
    for value in mid:
        yield from build_lines('├─', '│ ', value)
    yield from build_lines('└─', '  ', end)

def build_lines(first, other, values):
    try:
        yield first + next(values)
        for value in values:
            yield other + value
    except StopIteration:
        return

In [ ]:
def main(args):
    if len(args) > 0:
        model = parse_source(args[0])
        if model:
            print(build_tree(model))

Vamos ignorar os avisos por enquanto e tentar analisar uma sentença válida:

In [ ]:
main(["a = 3 + 4 * 5 ;"])

Nada aconteceu! Vamos ver o que acontece com uma sentença inválida:

In [ ]:
main(["a == 3 ;"])

Agora vamos adicionar informações para construir uma árvore de sintaxe:

In [ ]:
# Fake token to signal "end of file"
class EOF:
    type = 'EOF'
    value = 'EOF'
    lineno = 'EOF'
    index = -1
    end = -1

class Tokens:

    def __init__(self, tokens):
        self.tokens = tokens
        self._lookahead = None

    @property
    def lookahead(self):
        if self._lookahead is None:
            self._lookahead = next(self.tokens, EOF)    # <<< Fail at the "end of file"
        return self._lookahead

    def peek(self, *types):
        # Returns the current token, but without consuming it.
        if self.lookahead.type in types:
            return self.lookahead
        else:
            return None

    def accept(self, *types):
        # Accept the next token if it has a given type, otherwise return None (not an error)
        tok = self.peek(*types)
        if tok:
            self._lookahead = None
        return tok

    def expect(self, *types):
        # Requires the next token to have toktype or SyntaxError
        tok = self.accept(*types)
        if not tok:
            self.error()
        return tok

    def error(self):
        # Default error handling function.
        token = self.lookahead
        if token:
            lineno = getattr(token, 'lineno', 0)
            if lineno:
                raise SyntaxError(f'Syntax error at line {lineno}, token={token.type}\n')
            else:
                raise SyntaxError(f'Syntax error, token={token.type}')
        else:
            raise SyntaxError('Parse error in input. EOF\n')

# <program> ::= <statements> EOF
def parse_program(tokens):
    statements = parse_statements(tokens)
    return ('program', statements)

# <statements> ::= { <statement> }+
def parse_statements(tokens):
    statements = [ parse_statement(tokens) ]
    while tokens.peek('PRINT', 'ID', 'IF'):
        statements += [ parse_statement(tokens) ]
    return statements

# <statement> ::= <print_statement>
#               | <assign_statement>
#               | <if_statement>
def parse_statement(tokens):
    if tokens.peek('PRINT'):
        return parse_print_statement(tokens)
    elif tokens.peek('ID'):
        return parse_assignment_statement(tokens)
    elif tokens.peek('IF'):
        return parse_if_statement(tokens)
    else:
        tokens.error()

# <print_statement> ::= PRINT <expression> SEMI
def parse_print_statement(tokens):
    tokens.expect('PRINT')
    expression = parse_expression(tokens)
    tokens.expect('SEMI')
    return ('print', expression)

# <assign_statement> ::= <identifier> EQUALS <expression> SEMI
def parse_assignment_statement(tokens):
    identifier = parse_identifier(tokens)
    tokens.expect('EQUALS')
    expression = parse_expression(tokens)
    tokens.expect('SEMI')
    return ('assign', identifier, expression)

# <if_statement> ::= IF <expression> LBRACE <statements> RBRACE { ELSE LBRACE <statements> RBRACE }?
def parse_if_statement(tokens):
    tokens.expect('IF')
    test = parse_expression(tokens)
    tokens.expect('LBRACE')
    consequence = parse_statements(tokens)
    tokens.expect('RBRACE')

    if tokens.accept('ELSE'):
        tokens.expect('LBRACE')
        alternative = parse_statements(tokens)
        tokens.expect('RBRACE')
    else:
        alternative = None

    return ('if', test, consequence, alternative)

# <expression> ::= <primary_expression> { (PLUS|TIMES|EQ|NE) <primary_expression> }*
def parse_expression(tokens):
    left = parse_primary_expression(tokens)
    while op := tokens.accept('PLUS', 'TIMES', 'EQ', 'NE'):
        right = parse_primary_expression(tokens)
        left = (f'expr: {op.value}', left, right)
    return left

# <primary_expression> ::= <constant>
#                        | <identifier>
#                        | LPAREN <expression> RPAREN
def parse_primary_expression(tokens):
    if tokens.peek('NUM'):
        return parse_constant(tokens)
    elif tokens.peek('ID'):
        return parse_identifier(tokens)
    elif tokens.accept('LPAREN'):
        expression = parse_expression(tokens)
        tokens.expect('RPAREN')
        return expression
    else:
        tokens.error()

# <constant> ::= NUM
def parse_constant(tokens):
    constant = tokens.expect('NUM')
    return (f'num: {constant.value}')

# <identifier> ::= ID
def parse_identifier(tokens):
    identifier = tokens.expect('ID')
    return (f'id: {identifier.value}')

# High level function that takes input tokens and turns it into a syntax tree.
# This is a natural place to use some kind of generator function.

def parse_tokens(raw_tokens):
    try:
        tokens = Tokens(raw_tokens)
        return parse_program(tokens)
    except SyntaxError as e:
        print(e)

In [ ]:
main(["a = 3 + 4 * 5 ;"])

Construída a árvore de sintaxe, notamos que ela não está respeitando a precedência dos operadores. Ou seja, ela retorna (3 + 4) * 5 em vez de 3 + (4 * 5).

Para impor a precedência e a associatividade dos operadores no analisador sintático, podemos estruturar a gramática em uma hierarquia de regras. No SLY, esse problema pode ser facilmente resolvido adicionando a variável `precedence` à classe do analisador. Para um melhor entendimento, consulte a seção [“Lidando com Gramáticas Ambíguas”](https://sly.readthedocs.io/en/latest/sly.html#dealing-with-ambiguous-grammars)
da documentação do SLY.

Vejamos como podemos modificar a gramática para forçar o analisador sintático a priorizar os tokens `TIMES` em relação aos tokens `PLUS`:

In [ ]:
# Fake token to signal "end of file"
class EOF:
    type = 'EOF'
    value = 'EOF'
    lineno = 'EOF'
    index = -1
    end = -1

class Tokens:

    def __init__(self, tokens):
        self.tokens = tokens
        self._lookahead = None

    @property
    def lookahead(self):
        if self._lookahead is None:
            self._lookahead = next(self.tokens, EOF)    # <<< Fail at the "end of file"
        return self._lookahead

    def peek(self, *types):
        # Returns the current token, but without consuming it.
        if self.lookahead.type in types:
            return self.lookahead
        else:
            return None

    def accept(self, *types):
        # Accept the next token if it has a given type, otherwise return None (not an error)
        tok = self.peek(*types)
        if tok:
            self._lookahead = None
        return tok

    def expect(self, *types):
        # Requires the next token to have toktype or SyntaxError
        tok = self.accept(*types)
        if not tok:
            self.error()
        return tok

    def error(self):
        # Default error handling function.
        token = self.lookahead
        if token:
            lineno = getattr(token, 'lineno', 0)
            if lineno:
                raise SyntaxError(f'Syntax error at line {lineno}, token={token.type}\n')
            else:
                raise SyntaxError(f'Syntax error, token={token.type}')
        else:
            raise SyntaxError('Parse error in input. EOF\n')

# <program> ::= <statements> EOF
def parse_program(tokens):
    statements = parse_statements(tokens)
    return ('program', statements)

# <statements> ::= { <statement> }+
def parse_statements(tokens):
    statements = [ parse_statement(tokens) ]
    while tokens.peek('PRINT', 'ID', 'IF'):
        statements += [ parse_statement(tokens) ]
    return statements

# <statement> ::= <print_statement>
#               | <assign_statement>
#               | <if_statement>
def parse_statement(tokens):
    if tokens.peek('PRINT'):
        return parse_print_statement(tokens)
    elif tokens.peek('ID'):
        return parse_assignment_statement(tokens)
    elif tokens.peek('IF'):
        return parse_if_statement(tokens)
    else:
        tokens.error()

# <print_statement> ::= PRINT <expression> SEMI
def parse_print_statement(tokens):
    tokens.expect('PRINT')
    expression = parse_expression(tokens)
    tokens.expect('SEMI')
    return ('print', expression)

# <assign_statement> ::= <identifier> EQUALS <expression> SEMI
def parse_assignment_statement(tokens):
    identifier = parse_identifier(tokens)
    tokens.expect('EQUALS')
    expression = parse_expression(tokens)
    tokens.expect('SEMI')
    return ('assign', identifier, expression)

# <if_statement> ::= IF <expression> LBRACE <statements> RBRACE { ELSE LBRACE <statements> RBRACE }?
def parse_if_statement(tokens):
    tokens.expect('IF')
    test = parse_expression(tokens)
    tokens.expect('LBRACE')
    consequence = parse_statements(tokens)
    tokens.expect('RBRACE')

    if tokens.accept('ELSE'):
        tokens.expect('LBRACE')
        alternative = parse_statements(tokens)
        tokens.expect('RBRACE')
    else:
        alternative = None

    return ('if', test, consequence, alternative)

# <expression> ::= <addition_expression> { (EQ|NE) <addition_expression> }*
def parse_expression(tokens):
    left = parse_addition_expression(tokens)
    while op := tokens.accept('EQ', 'NE'):
        right = parse_addition_expression(tokens)
        left = (f'expr: {op.value}', left, right)
    return left

# <addition_expression> ::= <multiply_expression> { PLUS <multiply_expression> }*
def parse_addition_expression(tokens):
    left = parse_multiply_expression(tokens)
    while op := tokens.accept('PLUS'):
        right = parse_multiply_expression(tokens)
        left = (f'expr: {op.value}', left, right)
    return left

# <multiply_expression> = <primary_expression> { TIMES <primary_expression> }*
def parse_multiply_expression(tokens):
    left = parse_primary_expression(tokens)
    while op := tokens.accept('TIMES'):
        right = parse_primary_expression(tokens)
        left = (f'expr: {op.value}', left, right)
    return left

# <primary_expression> ::= <constant>
#                        | <identifier>
#                        | LPAREN <expression> RPAREN
def parse_primary_expression(tokens):
    if tokens.peek('NUM'):
        return parse_constant(tokens)
    elif tokens.peek('ID'):
        return parse_identifier(tokens)
    elif tokens.accept('LPAREN'):
        expression = parse_expression(tokens)
        tokens.expect('RPAREN')
        return expression
    else:
        tokens.error()

# <constant> ::= NUM
def parse_constant(tokens):
    constant = tokens.expect('NUM')
    return (f'num: {constant.value}')

# <identifier> ::= ID
def parse_identifier(tokens):
    identifier = tokens.expect('ID')
    return (f'id: {identifier.value}')

# High level function that takes input tokens and turns it into a syntax tree.
# This is a natural place to use some kind of generator function.

def parse_tokens(raw_tokens):
    try:
        tokens = Tokens(raw_tokens)
        return parse_program(tokens)
    except SyntaxError as e:
        print(e)

Para resolver o problema, a regra `expression` foi quebrada em quatro camadas:

```
<expression> ::= <addition_expression> { (EQ|NE) <addition_expression> }*

<addition_expression> ::= <multiply_expression> { PLUS <multiply_expression> }*

<multiply_expression> = <primary_expression> { TIMES <primary_expression> }*

<primary_expression> ::= <constant>
                       | <identifier>
                       | LPAREN <expression> RPAREN
```

Vamos executar o analisador sintático para nossas sentenças de exemplo novamente:

In [ ]:
main(["a = 3 + 4 * 5 ;"])

Outros exemplos:

In [ ]:
main(["a = 3 * ;"])

In [ ]:
main(["a == 3 ; "])

In [ ]:
main(["print a ;"])

In [ ]:
code = '''
a = 3;
b = 4 * a;
print (a+b);
'''

In [ ]:
main([code])

In [ ]:
code = '''
a = 7;
b = 10;
c = 5;

if a*a == b*b + c*c {
   print 1;
} else {
   print 0;
}
'''

In [ ]:
main([code])

## Escrevendo um Analisador Sintático para a Linguagem uZig

Nesta etapa, você deve escrever uma versão preliminar de um analisador sintático para a linguagem uZig. A especificação da gramática do uZig em BNF está [aqui](https://colab.research.google.com/drive/12AJRHYnY_Mtlr0aRb_7i0D-3_PHoscgc?usp=sharing).

### Especificação
Sua tarefa é traduzir as regras listadas em uma gramática BNF em uma coleção de rotinas em Python. Assim, uma regra gramatical como:

```
    <program> ::= <statement_list> EOF
```

torna-se uma função do Python da forma:

```python
# <program> ::= <statement_list> EOF
def parse_program(tokens):
    statement_list = parse_statement_list(tokens)
    return ('program', statement_list)
```

No SLY, em vez de rotinas em Python, você deve escrever métodos de classe decorados pelo decorador `@_()`. O nome de cada método deve corresponder ao nome da regra gramatical que está sendo analisada. O argumento para o decorador `@_()` é uma cadeia de caracteres que descreve o lado direito da gramática. Para um melhor entendimento, estude o capítulo [“Escrevendo um Analisador Sintático”](https://sly.readthedocs.io/en/latest/sly.html#writing-a-parser) da documentação do SLY. Assim, a regra gramatical acima torna-se um método de classe do Python da forma:

```python
class UZigParser(Parser):
    """A parser for the uZig language."""
    ...
    # <program> ::= <statement_list> EOF
    @_('statement_list')
    def program(self, p):
        return ('program', p.statement_list)
```

Para construir uma árvore de sintaxe, basta criar e retornar uma tupla ou lista em cada função de regra gramatical, como mostrado acima.

Seu objetivo, ao final deste segundo projeto, é reconhecer **sintaticamente** programas expressos na linguagem uZig.
Para isso, o ideal é que você faça com que sua gramática não apresente **nenhum** conflito.

**Sugestão:** Você deve começar de forma simples e trabalhar incrementalmente até construir a gramática completa.

### Esboço do Analisador Sintático

In [ ]:
!pip install sly

Copie o código do analisador léxico que você escreveu no [primeiro projeto](https://colab.research.google.com/drive/1JYNeH16cMf46jRF8JGYBRGeCQng8LZhO?usp=sharing) e cole na célula abaixo.

In [ ]:
# High level function that takes input source text and turns it into tokens.
# This is a natural place to use some kind of generator function.

def tokenize(text):
    ...
    yield tok
    ...

In [ ]:
# High level function that takes input tokens and turns it into a syntax tree.
# This is a natural place to use some kind of generator function.

def parse_tokens(tokens):
    ...

In [ ]:
# Top-level function that runs everything
def parse_source(text):
    tokens = tokenize(text)
    model = parse_tokens(tokens)     # You need to implement this part
    return model

In [ ]:
def build_tree(root):
    return '\n'.join(_build_tree(root))

def _build_tree(node):
    if isinstance(node, list):
        if not node: return
        node = tuple(node)

    if not isinstance(node, tuple):
        yield " "+str(node)
        return

    values = [_build_tree(n) for n in node]
    if len(values) == 1:
        yield from build_lines('──', '  ', values[0])
        return

    start, *mid, end = values
    yield from build_lines('┬─', '│ ', start)
    for value in mid:
        yield from build_lines('├─', '│ ', value)
    yield from build_lines('└─', '  ', end)

def build_lines(first, other, values):
    try:
        yield first + next(values)
        for value in values:
            yield other + value
    except StopIteration:
        return

In [ ]:
# Main program to test on input files
def main(filename):
    with open(filename) as file:
        text = file.read()

    model = parse_source(text)
    if model:
        print(build_tree(model))

## Teste
Para o desenvolvimento inicial, tente executar o analisador sintático em um arquivo de entrada de exemplo, como:

```
// print values of factorials
var n : i32 = 1;
var value : i32 = 1;

while (n < 10)
{
	value = value * n;
	@print(value);
	n = n + 1;
}
```

In [ ]:
%%file test.uzig
// print values of factorials
var n : i32 = 1;
var value : i32 = 1;

while (n < 10)
{
	value = value * n;
	@print(value);
	n = n + 1;
}

E o resultado será semelhante ao texto mostrado abaixo.

```
┬─ program
└─┬─┬─ variable: n
  │ ├─ type: i32
  │ └─ literal: i32, 1
  ├─┬─ variable: value
  │ ├─ type: i32
  │ └─ literal: i32, 1
  └─┬─ while
    ├─┬─ binary_op: <
    │ ├─ location: n
    │ └─ literal: i32, 10
    └─┬─ block
      └─┬─┬─ assignment
        │ ├─ location: value
        │ └─┬─ binary_op: *
        │   ├─ location: value
        │   └─ location: n
        ├─┬─ expression
        │ └─┬─ builtin: @print
        │   └─── location: value
        └─┬─ assignment
          ├─ location: n
          └─┬─ binary_op: +
            ├─ location: n
            └─ literal: i32, 1
```

In [ ]:
main(["test.uzig"])

Estude cuidadosamente a saída do analisador sintático e certifique-se de que ela faz sentido. Quando estiver razoavelmente satisfeito com a saída, tente executar alguns dos testes mais complicados projetados para testar vários cenários atípicos, fora do padrão esperado. Você pode usar como base os exemplos contidos [aqui](https://colab.research.google.com/drive/12AJRHYnY_Mtlr0aRb_7i0D-3_PHoscgc?usp=sharing).

No [AVA](https://ava.ufscar.br/mod/quiz/view.php?id=1035052) há um grande conjunto de testes para verificar seu código: confira-os para ver mais exemplos.

## Envie seu trabalho
Depois de concluir esta tarefa, salve o código do seu [analisador léxico](#scrollTo=78p9-xkBSKbs) em um arquivo chamado `uzig_lexer.py`, copie o código do seu [analisador sintático](#scrollTo=eeQIg6Oh4-il) e o submeta no [AVA](https://ava.ufscar.br/mod/quiz/view.php?id=1035052).

## Anexo
A lista abaixo define os nós da árvore de sintaxe que devem ser retornados em cada regra da gramática:

```
program = tuple('program', statement_list)

statement_list = list(statement)

statement = assignment_statement
          | variable_definition
          | const_definition
          | if_statement
          | while_statement
          | break_statement
          | continue_statement
          | expression_statement

assignment_statement = tuple('assignment', location, expression)

variable_definition = tuple('variable: ' + str(IDENTIFIER), type, expression)

const_definition = tuple('const: ' + str(IDENTIFIER), type, expression)

if_statement = tuple('if', expression, block0, block1)

while_statement = tuple('while', expression, block)

break_statement = tuple('break')

continue_statement = tuple('continue')

expression_statement = tuple('expression', expression)

block = tuple('block', statement_list)

expression = tuple('binary_op: +', expression0, expression1)
           | tuple('binary_op: -', expression0, expression1)
           | tuple('binary_op: *', expression0, expression1)
           | tuple('binary_op: /', expression0, expression1)
           | tuple('binary_op: %', expression0, expression1)
           | tuple('binary_op: <=', expression0, expression1)
           | tuple('binary_op: <', expression0, expression1)
           | tuple('binary_op: >=', expression0, expression1)
           | tuple('binary_op: >', expression0, expression1)
           | tuple('binary_op: ==', expression0, expression1)
           | tuple('binary_op: !=', expression0, expression1)
           | tuple('binary_op: and', expression0, expression1)
           | tuple('binary_op: or', expression0, expression1)
           | tuple('unary_op: +', expression)
           | tuple('unary_op: -', expression)
           | tuple('unary_op: !', expression)
           | literal
           | location
           | tuple('builtin: @print', expression_list)
           | expression

literal = tuple('literal: u8, ' + str(CHAR_LITERAL))
        | tuple('literal: f64, ' + str(FLOAT))
        | tuple('literal: i32, ' + str(INTEGER))
        | tuple('literal: []const u8, ' + str(STRINGLITERAL))        
        | tuple('literal: bool, true')
        | tuple('literal: bool, false')

expression_list = list(expression)

location = tuple('location: ' + str(IDENTIFIER))

type = tuple('type: ' + str(IDENTIFIER))
```

Um novo nó é criado sempre que o valor retornado for uma tupla do Python, por exemplo:

```python
    # <program> ::= <statements> EOF
    def parse_program(tokens):
        statements = parse_statements(tokens)
        return ('program', statements)
```

Uma lista de nós é criada sempre que o valor retornado for uma lista do Python, por exemplo:

```python
    # <statements> ::= { <statement> }+
    def parse_statements(tokens):
        statements = [ parse_statement(tokens) ]
        while tokens.peek('PRINT', 'ID', 'IF'):
            statements += [ parse_statement(tokens) ]
        return statements
```

Uma referência para um nó é criada sempre que o valor retornado em uma regra for o nome de outra regra.